# LeetCode 2383 — Minimum Hours of Training to Win a Competition bằng Python

Link đề: https://leetcode.com/problems/minimum-hours-of-training-to-win-a-competition/ · Độ khó: **Easy** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minNumberOfHours(self, initialEnergy: int, initialExperience: int, energy: List[int], experience: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **mô phỏng một chuỗi trận đấu** trong đó mỗi trận đòi hỏi hai chỉ số (năng lượng và kinh nghiệm) **lớn hơn hẳn** (strictly greater) đối thủ.
2. Nhận ra hai chỉ số **độc lập** với nhau: năng lượng chỉ giảm, kinh nghiệm chỉ tăng, nên có thể tính riêng từng phần rồi cộng lại.
3. Dùng `zip`, `sum`, `max`, `itertools.accumulate` để viết lời giải ngắn gọn và đúng.
4. Viết 4 cách: duyệt mọi cặp giờ tập `(a, b)`, mô phỏng từng giờ một, tham lam bù phần thiếu đúng lúc cần, và công thức đóng (tổng + prefix); rồi đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi kinh điển: quên dấu "lớn hơn **hẳn**" (dùng `>=`), quên `+ 1`, quên `max(0, ...)`, và quên cập nhật chỉ số sau mỗi trận.

## 2. Đề bài & ví dụ

Bạn tham gia một giải đấu. Ban đầu bạn có `initialEnergy` (năng lượng) và `initialExperience` (kinh nghiệm), đều là số nguyên dương. Có `n` đối thủ, đối thủ thứ `i` có `energy[i]` và `experience[i]`. Bạn đấu **lần lượt** theo thứ tự `0, 1, ..., n-1`.

Để thắng đối thủ `i`, **cả hai** điều kiện phải đúng **chặt**:

- năng lượng hiện tại `>` `energy[i]`,
- kinh nghiệm hiện tại `>` `experience[i]`.

Sau khi thắng: kinh nghiệm **tăng** `experience[i]`, năng lượng **giảm** `energy[i]`.

Trước giải đấu, bạn được tập luyện một số giờ. Mỗi giờ tập bạn chọn **tăng 1 đơn vị năng lượng ban đầu** hoặc **tăng 1 đơn vị kinh nghiệm ban đầu**. Trả về **số giờ tập tối thiểu** để thắng cả `n` đối thủ.

| Ví dụ | initialEnergy | initialExperience | energy | experience | Output |
|---|---|---|---|---|---|
| 1 | `5` | `3` | `[1,4,3,2]` | `[2,6,3,1]` | `8` |
| 2 | `2` | `4` | `[1]` | `[3]` | `0` |

Giải thích ví dụ 1: tập 6 giờ cho năng lượng (thành 11) và 2 giờ cho kinh nghiệm (thành 5), tổng 8 giờ; khi đó thắng cả bốn đối thủ. Ví dụ 2: `2 > 1` và `4 > 3`, không cần tập.

Constraints: `n == energy.length == experience.length`; `1 <= n <= 100`; `1 <= initialEnergy, initialExperience, energy[i], experience[i] <= 100`.

In [1]:
initialEnergy, initialExperience = 5, 3
energy, experience = [1, 4, 3, 2], [2, 6, 3, 1]
e, x = initialEnergy, initialExperience
for i in range(len(energy)):
    print(f"trận {i}: e={e} (cần > {energy[i]}), x={x} (cần > {experience[i]})")
    e -= energy[i]
    x += experience[i]
print("Nếu không tập: ở trận 1 đã có e=4 không lớn hơn 4 -> thua; cần tập thêm.")

trận 0: e=5 (cần > 1), x=3 (cần > 2)
trận 1: e=4 (cần > 4), x=5 (cần > 6)
trận 2: e=0 (cần > 3), x=11 (cần > 3)
trận 3: e=-3 (cần > 2), x=14 (cần > 1)
Nếu không tập: ở trận 1 đã có e=4 không lớn hơn 4 -> thua; cần tập thêm.


## 3. Kiến thức nền

### 3.1. `zip` để duyệt hai danh sách song song

`zip(a, b)` ghép phần tử cùng chỉ số của hai danh sách. Ở đây `zip(energy, experience)` cho ta từng cặp `(năng lượng đối thủ, kinh nghiệm đối thủ)` theo thứ tự đấu.

In [2]:
energy, experience = [1, 4, 3, 2], [2, 6, 3, 1]
for en, ex in zip(energy, experience):
    print("đối thủ:", en, ex)
print(list(zip(energy, experience)))

đối thủ: 1 2
đối thủ: 4 6
đối thủ: 3 3
đối thủ: 2 1
[(1, 2), (4, 6), (3, 3), (2, 1)]


### 3.2. Điều kiện "lớn hơn hẳn" và phần thiếu

Nếu mình có `cur` và cần `cur > need`, tức `cur >= need + 1`. Phần thiếu là `max(0, need + 1 - cur)`. Hàm `max(0, ...)` rất quan trọng: nếu đã dư thì phần thiếu là `0`, **không** được âm.

In [3]:
def thieu(cur, need):
    return max(0, need + 1 - cur)

print(thieu(4, 4))   # 4 > 4 sai, thiếu 1
print(thieu(5, 4))   # đã thắng, thiếu 0
print(thieu(1, 10))  # thiếu 10

1
0
10


### 3.3. `sum` và tổng tiền tố với `itertools.accumulate`

`sum(lst)` là tổng. `accumulate(lst)` sinh dãy tổng dồn: `[a, a+b, a+b+c, ...]`. Tổng của **các phần tử trước chỉ số `i`** là tổng tiền tố đã lùi một bước, rất tiện để biết "kinh nghiệm đã tích lũy trước trận `i`".

In [4]:
from itertools import accumulate
experience = [2, 6, 3, 1]
pref = list(accumulate(experience))
print("tổng dồn:", pref)
before = [0] + pref[:-1]
print("tổng trước trận i:", before)
print(sum(experience))

tổng dồn: [2, 8, 11, 12]
tổng trước trận i: [0, 2, 8, 11]
12


### 3.4. Mô phỏng bằng vòng lặp `while`

Vòng `while cur <= need: cur += 1` tăng từng đơn vị đến khi hợp lệ. Cách này chậm nếu thiếu nhiều nhưng rất dễ hiểu; giá trị ở đề chỉ tối đa 100 nên vẫn chạy nhanh.

In [5]:
cur, need, hours = 4, 4, 0
while cur <= need:
    cur += 1
    hours += 1
print(cur, hours)

5 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy tính tay ví dụ 1 và để ý **năng lượng giảm dần** còn **kinh nghiệm tăng dần**.

- Năng lượng: ban đầu `E`. Trước trận 0 cần `E > 1`. Sau trận 0 còn `E - 1`; trước trận 1 cần `E - 1 > 4`, tức `E > 5`. Sau trận 1 còn `E - 5`; trước trận 2 cần `E - 5 > 3`, tức `E > 8`. Trước trận 3 cần `E - 8 > 2`, tức `E > 10`. Điều kiện **chặt nhất** là `E > 10` (trận cuối) nên `E >= 11`. Có `5` nên tập `6` giờ.
- Kinh nghiệm: ban đầu `X = 3`. Trước trận 0 cần `X > 2` đúng. Sau trận 0 có `X + 2`; trước trận 1 cần `X + 2 > 6`, tức `X > 4`. Sau trận 1 có `X + 8`; cần `> 3` luôn đúng với `X >= 1`. Vậy cần `X >= 5`, tập `2` giờ.

Hai phần **không ảnh hưởng nhau**, nên đáp án là `6 + 2 = 8`.

Trực giác quan trọng: *năng lượng chỉ giảm nên điều kiện chặt nhất rơi vào trận cuối*; *kinh nghiệm chỉ tăng nên phải kiểm tra từng trận để tìm trận chặt nhất*.

In [6]:
E0, X0 = 5, 3
energy, experience = [1, 4, 3, 2], [2, 6, 3, 1]
sum_e = 0
for i, en in enumerate(energy):
    sum_e += en
    print(f"trận {i}: cần E > {sum_e}  => E >= {sum_e + 1}")
print("giờ tập năng lượng:", max(0, sum(energy) + 1 - E0))
sum_x = 0
need_x = 0
for i, ex in enumerate(experience):
    need = ex + 1 - sum_x          # X >= ex + 1 - (kinh nghiệm đã tích lũy)
    need_x = max(need_x, need)
    print(f"trận {i}: cần X >= {need}")
    sum_x += ex
print("giờ tập kinh nghiệm:", max(0, need_x - X0))

trận 0: cần E > 1  => E >= 2
trận 1: cần E > 5  => E >= 6
trận 2: cần E > 8  => E >= 9
trận 3: cần E > 10  => E >= 11
giờ tập năng lượng: 6
trận 0: cần X >= 3
trận 1: cần X >= 5
trận 2: cần X >= -4
trận 3: cần X >= -9
giờ tập kinh nghiệm: 2


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp giờ tập `(a, b)`

**Ý tưởng:** tổng số giờ tập `h = a + b`, trong đó `a` giờ cho năng lượng, `b` giờ cho kinh nghiệm. Ta thử `h = 0, 1, 2, ...`; với mỗi `h` thử mọi cách chia `a = 0..h`. Cặp đầu tiên cho phép **mô phỏng thắng cả giải** thì `h` là đáp án nhỏ nhất.

Hàm `win(E, X)` chỉ việc mô phỏng đúng đề bài.

In [7]:
from typing import List

class SolutionBrute:
    def minNumberOfHours(self, initialEnergy: int, initialExperience: int, energy: List[int], experience: List[int]) -> int:
        def win(e, x):
            for en, ex in zip(energy, experience):
                if e <= en or x <= ex:
                    return False
                e -= en
                x += ex
            return True

        h = 0
        while True:
            for a in range(h + 1):
                if win(initialEnergy + a, initialExperience + (h - a)):
                    return h
            h += 1

In [8]:
sol = SolutionBrute()
print(sol.minNumberOfHours(5, 3, [1, 4, 3, 2], [2, 6, 3, 1]))  # 8
print(sol.minNumberOfHours(2, 4, [1], [3]))                    # 0

8
0


**Độ phức tạp:** nếu đáp án là `H`, ta thử khoảng `H^2 / 2` cặp, mỗi cặp mô phỏng `O(n)`, tổng `O(H^2 · n)`. Với `n, giá trị <= 100`, `H` có thể lên tới ~10^4, nên brute quá chậm. Ta chỉ dùng nó làm "máy kiểm tra" cho test nhỏ. **Nút thắt:** ta thử cả những cách chia giờ hiển nhiên vô ích; trong khi hai chỉ số độc lập nên lẽ ra có thể tính riêng.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** hai nguồn lực không "đổi" cho nhau. Việc thiếu năng lượng không thể bù bằng kinh nghiệm và ngược lại. Vì vậy:

- **Invariant năng lượng:** trước trận `i` năng lượng là `E - (energy[0] + ... + energy[i-1])`. Điều kiện thắng là `E > energy[0] + ... + energy[i]` (tổng tiền tố **bao gồm** `i`). Tổng tiền tố tăng dần nên điều kiện chặt nhất ở trận cuối: `E >= sum(energy) + 1`.
- **Invariant kinh nghiệm:** trước trận `i` kinh nghiệm là `X + (experience[0] + ... + experience[i-1])`. Điều kiện: `X > experience[i] - (tổng trước i)`.

Quy tắc "nếu… thì…" cho cách tham lam: **nếu** ngay trước trận `i` mà một chỉ số chưa vượt đối thủ **thì** bù đúng phần thiếu (cộng đúng `need + 1 - cur`). Vì tập trước giải đấu tương đương tập "đúng lúc cần" (chỉ số nào cũng chỉ tăng thêm khi tập), nên bù muộn nhất là bù ít nhất.

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng và tập từng giờ một

Duyệt từng đối thủ. Nếu năng lượng chưa đủ, tăng năng lượng **1 đơn vị** và đếm 1 giờ; lặp đến khi đủ. Tương tự cho kinh nghiệm. Sau đó áp dụng kết quả trận đấu.

In [9]:
from typing import List

class SolutionSimulate:
    def minNumberOfHours(self, initialEnergy: int, initialExperience: int, energy: List[int], experience: List[int]) -> int:
        e, x, hours = initialEnergy, initialExperience, 0
        for en, ex in zip(energy, experience):
            while e <= en:
                e += 1
                hours += 1
            while x <= ex:
                x += 1
                hours += 1
            e -= en
            x += ex
        return hours

In [10]:
print(SolutionSimulate().minNumberOfHours(5, 3, [1, 4, 3, 2], [2, 6, 3, 1]))  # 8
print(SolutionSimulate().minNumberOfHours(2, 4, [1], [3]))                    # 0

8
0


**Độ phức tạp:** mỗi giờ tập là một lần lặp `while`, tổng số lần lặp bằng đáp án `H` cộng `n`. Time `O(n + H)` với `H <= ~10^4`, Space `O(1)`.

### Cách 3 — Tham lam bù đúng phần thiếu (không lặp từng đơn vị)

Thay vì tăng từng đơn vị, tính ngay `need + 1 - cur` và cộng một lần. Đây là bản "tính thẳng" của cách 2.

In [11]:
from typing import List

class SolutionGreedy:
    def minNumberOfHours(self, initialEnergy: int, initialExperience: int, energy: List[int], experience: List[int]) -> int:
        e, x, hours = initialEnergy, initialExperience, 0
        for en, ex in zip(energy, experience):
            if e <= en:
                gap = en + 1 - e
                hours += gap
                e += gap
            if x <= ex:
                gap = ex + 1 - x
                hours += gap
                x += gap
            e -= en
            x += ex
        return hours

In [12]:
print(SolutionGreedy().minNumberOfHours(5, 3, [1, 4, 3, 2], [2, 6, 3, 1]))  # 8
print(SolutionGreedy().minNumberOfHours(2, 4, [1], [3]))                    # 0

8
0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Công thức đóng: tổng và tổng tiền tố

Dùng đúng hai invariant ở mục 6:

- Giờ năng lượng `= max(0, sum(energy) + 1 - initialEnergy)`.
- Giờ kinh nghiệm `= max(0, max_i (experience[i] + 1 - (initialExperience + tổng experience trước i)))`.

In [13]:
from typing import List
from itertools import accumulate

class SolutionFormula:
    def minNumberOfHours(self, initialEnergy: int, initialExperience: int, energy: List[int], experience: List[int]) -> int:
        hours_energy = max(0, sum(energy) + 1 - initialEnergy)
        before = [0] + list(accumulate(experience))[:-1]   # kinh nghiệm tích lũy trước mỗi trận
        need = max(ex + 1 - (initialExperience + b) for ex, b in zip(experience, before))
        return hours_energy + max(0, need)

In [14]:
print(SolutionFormula().minNumberOfHours(5, 3, [1, 4, 3, 2], [2, 6, 3, 1]))  # 8
print(SolutionFormula().minNumberOfHours(2, 4, [1], [3]))                    # 0

8
0


**Độ phức tạp:** Time `O(n)` (`sum`, `accumulate` và một lượt `max`), Space `O(n)` cho danh sách tiền tố (có thể giảm xuống `O(1)` bằng cách cộng dồn thủ công).

## 8. TRACE từng bước

Bảng trạng thái cho ví dụ 1 theo **Cách 3 (tham lam)**, `E = 5, X = 3`:

| Trận | e trước | x trước | thiếu e | thiếu x | hours | e sau | x sau |
|---|---|---|---|---|---|---|---|
| 0 (1,2) | 5 | 3 | 0 | 0 | 0 | 4 | 5 |
| 1 (4,6) | 4 | 5 | 1 | 2 | 3 | 1 | 13 |
| 2 (3,3) | 1 | 13 | 3 | 0 | 6 | 1 | 16 |
| 3 (2,1) | 1 | 16 | 2 | 0 | 8 | 1 | 17 |

Cột "thiếu" là phần cần tập thêm để thắng chặt. Tổng `hours = 8`.

In [15]:
def trace_greedy(E, X, energy, experience):
    e, x, hours = E, X, 0
    for i, (en, ex) in enumerate(zip(energy, experience)):
        print(f"--- trận {i}: đối thủ (en={en}, ex={ex}); trước: e={e}, x={x}, hours={hours}")
        if e <= en:
            gap = en + 1 - e
            print(f"    năng lượng thiếu {gap} -> tập {gap} giờ")
            hours += gap
            e += gap
        if x <= ex:
            gap = ex + 1 - x
            print(f"    kinh nghiệm thiếu {gap} -> tập {gap} giờ")
            hours += gap
            x += gap
        e -= en
        x += ex
        print(f"    sau trận: e={e}, x={x}, hours={hours}")
    return hours

print("kết quả:", trace_greedy(5, 3, [1, 4, 3, 2], [2, 6, 3, 1]))

--- trận 0: đối thủ (en=1, ex=2); trước: e=5, x=3, hours=0
    sau trận: e=4, x=5, hours=0
--- trận 1: đối thủ (en=4, ex=6); trước: e=4, x=5, hours=0
    năng lượng thiếu 1 -> tập 1 giờ
    kinh nghiệm thiếu 2 -> tập 2 giờ
    sau trận: e=1, x=13, hours=3
--- trận 2: đối thủ (en=3, ex=3); trước: e=1, x=13, hours=3
    năng lượng thiếu 3 -> tập 3 giờ
    sau trận: e=1, x=16, hours=6
--- trận 3: đối thủ (en=2, ex=1); trước: e=1, x=16, hours=6
    năng lượng thiếu 2 -> tập 2 giờ
    sau trận: e=1, x=17, hours=8
kết quả: 8


## 9. Code nộp LeetCode

Dùng công thức đóng (ngắn, nhanh):

In [16]:
from typing import List
from itertools import accumulate

class Solution:
    def minNumberOfHours(self, initialEnergy: int, initialExperience: int, energy: List[int], experience: List[int]) -> int:
        hours_energy = max(0, sum(energy) + 1 - initialEnergy)
        before = [0] + list(accumulate(experience))[:-1]
        need = max(ex + 1 - (initialExperience + b) for ex, b in zip(experience, before))
        return hours_energy + max(0, need)

In [17]:
print(Solution().minNumberOfHours(5, 3, [1, 4, 3, 2], [2, 6, 3, 1]))
print(Solution().minNumberOfHours(2, 4, [1], [3]))

8
0


## 10. Vì sao thuật toán đúng

1. **Tách độc lập.** Điều kiện thắng là hội của hai điều kiện, mỗi điều kiện chỉ phụ thuộc vào **một** chỉ số (năng lượng chỉ phụ thuộc `E`; kinh nghiệm chỉ phụ thuộc `X`), vì thắng trận không phụ thuộc cái còn lại. Do đó tối thiểu hóa `a + b` bằng cách tối thiểu `a` và `b` riêng rẽ.
2. **Năng lượng.** Trước trận `i`, năng lượng là `E - S_{i-1}` với `S_i = energy[0] + ... + energy[i]`. Cần `E - S_{i-1} > energy[i]` ⇔ `E > S_i`. Vì `energy[i] >= 1`, `S_i` tăng nghiêm ngặt nên `S_{n-1}` là lớn nhất. Vậy `E >= S_{n-1} + 1` là điều kiện cần và đủ cho **mọi** trận.
3. **Kinh nghiệm.** Trước trận `i`, kinh nghiệm là `X + P_{i-1}`, `P_i` là tổng tiền tố của `experience`. Cần `X + P_{i-1} > experience[i]`, vậy `X >= max_i(experience[i] + 1 - P_{i-1})` (và `X` ban đầu đã `>= 1`). Mỗi điều kiện là một cận dưới; điều kiện đồng thời là cận dưới lớn nhất.
4. **Tối thiểu.** Mỗi giờ tập chỉ tăng đúng 1 đơn vị, nên số giờ nhỏ nhất bằng tổng hai phần thiếu, mỗi phần là `max(0, cận - giá trị ban đầu)`.
5. Cách tham lam đúng vì tập "đúng lúc cần" cho cùng tổng số giờ như tập trước (chỉ số chỉ tăng thêm); nó hoàn toàn khớp với công thức.

## 11. Phân tích Time / Space Complexity

Đặt `n` là số đối thủ và `H` là đáp án (số giờ tập).

| Cách | Time | Space |
|---|---|---|
| Brute (mọi cặp `(a, b)`) | `O(H^2 · n)` | `O(1)` |
| Mô phỏng từng giờ | `O(n + H)` | `O(1)` |
| Tham lam bù đúng phần thiếu | `O(n)` | `O(1)` |
| Công thức đóng (accumulate) | `O(n)` | `O(n)` |

Với `n <= 100` và giá trị `<= 100`, mọi cách trừ brute đều chạy tức thì. Trường hợp xấu nhất: `H` khoảng `10^4` (năng lượng cần ~`100·100 + 1`).

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [18]:
sols = [SolutionBrute(), SolutionSimulate(), SolutionGreedy(), SolutionFormula(), Solution()]
cases = [
    ((5, 3, [1, 4, 3, 2], [2, 6, 3, 1]), 8),   # ví dụ 1
    ((2, 4, [1], [3]), 0),                     # ví dụ 2
    ((1, 1, [1], [1]), 2),                     # bằng nhau vẫn thua: cần 2 và 2 -> mỗi bên +1
    ((100, 100, [1], [1]), 0),                 # dư thừa, không cần tập
    ((5, 5, [4], [4]), 0),                     # 5 > 4 cả hai
    ((4, 5, [4], [4]), 1),                     # chỉ thiếu năng lượng 1
    ((5, 4, [4], [4]), 1),                     # chỉ thiếu kinh nghiệm 1
]
for args, want in cases:
    for s in sols:
        got = s.minNumberOfHours(*args)
        assert got == want, (type(s).__name__, args, got, want)
print("7 test cố định đều đúng với cả 5 cách")

7 test cố định đều đúng với cả 5 cách


Thêm test `(1, 1, [2, 2], [2, 2])`: năng lượng cần `E >= 5` nên tập `4` giờ; kinh nghiệm: trận 0 cần `X >= 3` (tập 2), sau đó `X = 5 > 2` ổn. Tổng `4 + 2 = 6`. Ta khẳng định giá trị này dưới đây và đối chiếu mọi cách.

In [19]:
for s in sols:
    assert s.minNumberOfHours(1, 1, [2, 2], [2, 2]) == 6
# n lớn nhất và giá trị lớn nhất
big_e = [100] * 100
big_x = [100] * 100
for s in sols[1:]:
    assert s.minNumberOfHours(1, 1, big_e, big_x) == 10100, s
print("giá trị lớn nhất:", Solution().minNumberOfHours(1, 1, big_e, big_x))

giá trị lớn nhất: 10100


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh dữ liệu đúng constraints: `1 <= n <= 100`, mọi giá trị trong `[1, 100]`. Brute chỉ chạy với dữ liệu nhỏ (n ≤ 4, giá trị ≤ 6) để khỏi quá chậm.

In [20]:
import random
random.seed(2383)

# Test nhỏ: có cả brute
for _ in range(300):
    n = random.randint(1, 4)
    E = random.randint(1, 6)
    X = random.randint(1, 6)
    energy = [random.randint(1, 6) for _ in range(n)]
    experience = [random.randint(1, 6) for _ in range(n)]
    results = {type(s).__name__: s.minNumberOfHours(E, X, energy, experience) for s in sols}
    assert len(set(results.values())) == 1, (E, X, energy, experience, results)
print("300 test nhỏ: 5 cách khớp nhau")

# Test lớn: bỏ brute, kiểm tra 4 cách còn lại
fast = sols[1:]
for _ in range(500):
    n = random.randint(1, 100)
    E = random.randint(1, 100)
    X = random.randint(1, 100)
    energy = [random.randint(1, 100) for _ in range(n)]
    experience = [random.randint(1, 100) for _ in range(n)]
    results = {type(s).__name__: s.minNumberOfHours(E, X, energy, experience) for s in fast}
    assert len(set(results.values())) == 1, (E, X, results)
    assert results["Solution"] >= 0
print("500 test lớn: 4 cách nhanh khớp nhau")

300 test nhỏ: 5 cách khớp nhau
500 test lớn: 4 cách nhanh khớp nhau


### 12.3. Phản ví dụ: kiểm tra đáp án thật sự là tối thiểu và đủ

Với đáp án `h` tìm được, ta kiểm tra bằng mô phỏng: tồn tại cách chia `a + b = h` thắng, và với `h - 1` thì **không** có cách chia nào thắng.

In [21]:
def can_win(E, X, energy, experience):
    for en, ex in zip(energy, experience):
        if E <= en or X <= ex:
            return False
        E -= en
        X += ex
    return True

random.seed(7)
for _ in range(200):
    n = random.randint(1, 5)
    E, X = random.randint(1, 8), random.randint(1, 8)
    energy = [random.randint(1, 8) for _ in range(n)]
    experience = [random.randint(1, 8) for _ in range(n)]
    h = Solution().minNumberOfHours(E, X, energy, experience)
    assert any(can_win(E + a, X + h - a, energy, experience) for a in range(h + 1))
    if h > 0:
        assert not any(can_win(E + a, X + h - 1 - a, energy, experience) for a in range(h))
print("đáp án vừa đủ và không thể ít hơn")

đáp án vừa đủ và không thể ít hơn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` thay vì "lớn hơn hẳn" (quên `+ 1`)

Đề yêu cầu **strictly greater**. Nếu viết `sum(energy) - initialEnergy` (không `+ 1`) thì thiếu đúng 1 giờ.

In [22]:
from typing import List

def sai_khong_cong_1(initialEnergy: int, initialExperience: int, energy: List[int], experience: List[int]) -> int:
    hours = max(0, sum(energy) - initialEnergy)          # SAI: thiếu + 1
    x = initialExperience
    for ex in experience:
        if x <= ex:
            hours += ex + 1 - x
            x = ex + 1
        x += ex
    return hours

dung = Solution().minNumberOfHours(5, 3, [1, 4, 3, 2], [2, 6, 3, 1])
sai = sai_khong_cong_1(5, 3, [1, 4, 3, 2], [2, 6, 3, 1])
print("đúng:", dung, "| sai:", sai)
assert dung != sai

đúng: 8 | sai: 7


**Vì sao sai:** có `E = 10` khi tổng năng lượng bằng `10`: trận cuối cần `năng lượng > 2` với `e = 2` (sai!). Công thức đúng là `sum(energy) + 1 - initialEnergy`. Code đúng đã có ở mục 9.

### Lỗi 2: Quên `max(0, ...)` khi đã dư năng lượng

Nếu `initialEnergy` rất lớn, `sum(energy) + 1 - initialEnergy` âm. Cộng số âm làm giảm đáp án.

In [23]:
def sai_khong_max(initialEnergy, initialExperience, energy, experience):
    hours_energy = sum(energy) + 1 - initialEnergy       # SAI: có thể âm
    x, hours_x = initialExperience, 0
    for ex in experience:
        if x <= ex:
            gap = ex + 1 - x
            hours_x += gap
            x += gap
        x += ex
    return hours_energy + hours_x

args = (100, 1, [1, 1], [2, 2])
print("đúng:", Solution().minNumberOfHours(*args), "| sai:", sai_khong_max(*args))
assert Solution().minNumberOfHours(*args) != sai_khong_max(*args)

đúng: 2 | sai: -95


**Vì sao sai:** `100` năng lượng dư nhiều so với cần `3`, kết quả năng lượng phải là `0`, không phải `-97`. Luôn bọc "phần thiếu" bằng `max(0, ...)`.

### Lỗi 3: Dùng `<` thay vì `<=` khi kiểm tra thiếu hụt

Điều kiện thiếu là `cur <= need` (không phải `cur < need`) vì bằng nhau vẫn **thua**.

In [24]:
def sai_so_sanh(initialEnergy, initialExperience, energy, experience):
    e, x, hours = initialEnergy, initialExperience, 0
    for en, ex in zip(energy, experience):
        if e < en:                      # SAI: phải là e <= en
            gap = en + 1 - e
            hours += gap
            e += gap
        if x < ex:                      # SAI: phải là x <= ex
            gap = ex + 1 - x
            hours += gap
            x += gap
        e -= en
        x += ex
    return hours

args = (4, 5, [4], [4])
print("đúng:", Solution().minNumberOfHours(*args), "| sai:", sai_so_sanh(*args))
assert Solution().minNumberOfHours(*args) != sai_so_sanh(*args)

đúng: 1 | sai: 0


**Vì sao sai:** với `e = 4` và đối thủ `energy = 4` ta chỉ **hòa** chứ chưa thắng; phải tập thêm 1. Dạng điều kiện tổng quát: `cur <= need` ⇒ thiếu.

### Lỗi 4: Không cập nhật chỉ số sau mỗi trận

Nếu bạn so sánh mỗi đối thủ với `initialEnergy` và `initialExperience` ban đầu (không trừ năng lượng đã dùng, không cộng kinh nghiệm đã nhận), kết quả sai.

In [25]:
def sai_khong_cap_nhat(initialEnergy, initialExperience, energy, experience):
    hours = 0
    for en, ex in zip(energy, experience):
        hours = max(hours, 0)
        hours += max(0, en + 1 - initialEnergy)          # SAI: dùng lại giá trị ban đầu
        hours += max(0, ex + 1 - initialExperience)
    return hours

args = (5, 3, [1, 4, 3, 2], [2, 6, 3, 1])
print("đúng:", Solution().minNumberOfHours(*args), "| sai:", sai_khong_cap_nhat(*args))
assert Solution().minNumberOfHours(*args) != sai_khong_cap_nhat(*args)

đúng: 8 | sai: 5


**Vì sao sai:** trận sau chịu ảnh hưởng của các trận trước: năng lượng đã bị trừ, kinh nghiệm đã được cộng. Luôn cập nhật `e -= en`, `x += ex` ngay sau mỗi trận (hoặc dùng tổng tiền tố).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "tối thiểu số lần tăng để thỏa một chuỗi điều kiện" và các điều kiện **không phụ thuộc nhau** → tách thành từng chỉ số, mỗi chỉ số một cận dưới.
- Một chỉ số **chỉ giảm** (năng lượng) → điều kiện chặt nhất ở cuối, dùng `sum`. Một chỉ số **chỉ tăng** (kinh nghiệm) → duyệt từng trận lấy `max` của cận dưới (hoặc mô phỏng bù đúng lúc).
- "Lớn hơn hẳn" luôn nhắc bạn cộng `1`; còn "tối thiểu hóa" nhắc bạn bọc bằng `max(0, ...)`.
- Kiểm tra nhanh bằng ví dụ có nhiều trận giống nhau (ví dụ `[1]` và `[1]`) để thấy chênh lệch `+ 1`.
- Nếu không chắc công thức, hãy viết hàm `can_win(E, X)` rồi thử bằng brute trên dữ liệu bé để so.

## 15. Biến thể của bài

1. **Có thể chọn thứ tự đấu:** sắp xếp đối thủ theo quy tắc nào để số giờ tập tối thiểu? (Gợi ý: bài toán lập lịch kiểu "cuối cùng của kinh nghiệm", cần trao đổi cặp.)
2. **Mỗi giờ tập tăng nhiều hơn 1** (ví dụ tăng 3 năng lượng hoặc 2 kinh nghiệm): dùng làm tròn lên `ceil(thiếu / bước)`.
3. **Điều kiện `>=` thay vì `>`:** bỏ `+ 1`.
4. **Đồng thời chỉ số thứ ba** (ví dụ vàng): cùng tư duy tách riêng.
5. **Trả về chia giờ tập:** `(giờ năng lượng, giờ kinh nghiệm)` thay vì tổng.
6. **Đòi hỏi năng lượng không bao giờ về 0 sau trận:** thay điều kiện thành `e - en > 0`, vẫn tương đương `e > en`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute (mọi cặp `(a, b)`) | thử `h = 0,1,2,..` và mọi cách chia | `O(H^2 · n)` | `O(1)` | chỉ để kiểm tra test nhỏ |
| 2. Mô phỏng từng giờ | `while cur <= need: cur += 1` | `O(n + H)` | `O(1)` | dễ hiểu nhất, đủ nhanh với đề này |
| 3. Tham lam bù phần thiếu | cộng thẳng `need + 1 - cur` | `O(n)` | `O(1)` | cách khuyến nghị khi phỏng vấn |
| 4. Công thức đóng | `sum(energy) + 1 - E` và `max` tiền tố | `O(n)` | `O(n)` | ngắn gọn, thể hiện hiểu bản chất |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, hãy đoán: `Solution().minNumberOfHours(1, 1, [1], [1])`.

**Bài 2 (tính tay).** `initialEnergy = 3`, `initialExperience = 2`, `energy = [2, 1]`, `experience = [1, 2]`. Tính số giờ cần tập (energy và experience riêng).

**Bài 3 (điền TODO).** Hoàn thành hàm `giờ_năng_lượng(E, energy)` trả về số giờ tập năng lượng.

In [26]:
def gio_nang_luong(E, energy):
    # TODO: trả về số giờ tối thiểu để E > tổng energy
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết hàm `gio_kinh_nghiem(X, experience)` bằng vòng lặp (không dùng `accumulate`), rồi ghép với bài 3 để thành lời giải hoàn chỉnh.

**Bài 5 (biến thể / mini challenge).** Mỗi giờ tập tăng `step_e` năng lượng **hoặc** `step_x` kinh nghiệm (không phải 1). Viết `min_hours_step(E, X, energy, experience, step_e, step_x)`. Ví dụ với `step_e = 2`, `step_x = 3`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: cả hai chỉ số bằng đối thủ, hòa vẫn thua. Bài 2: tổng năng lượng `3`, cần `E >= 4`.
- **Mức 2 (vừa):** Bài 3: `max(0, sum(energy) + 1 - E)`. Bài 4: duyệt từng trận, giữ `x`, nếu `x <= ex` thì bù `ex + 1 - x`.
- **Mức 3 (sâu):** Bài 5: số giờ cần cho phần thiếu `g` là `ceil(g / step)`, tính bằng `(g + step - 1) // step`; nhưng mỗi lần bù làm tăng nhiều hơn mức cần (dư), nên cập nhật `x` bằng đúng số đã tăng.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1: 1 > 1 sai ở cả hai -> mỗi chỉ số +1 -> tổng 2
assert Solution().minNumberOfHours(1, 1, [1], [1]) == 2

# Bài 2: E cần > 2+1 = 3 -> E >= 4 (thiếu 1); X: trận 0 X>1 ổn -> X=3; trận 1 X>2 ổn -> không thiếu
assert Solution().minNumberOfHours(3, 2, [2, 1], [1, 2]) == 1

# Bài 3
def gio_nang_luong(E, energy):
    return max(0, sum(energy) + 1 - E)

# Bài 4
def gio_kinh_nghiem(X, experience):
    hours = 0
    for ex in experience:
        if X <= ex:
            gap = ex + 1 - X
            hours += gap
            X += gap
        X += ex
    return hours

# Bài 5
def min_hours_step(E, X, energy, experience, step_e, step_x):
    hours = 0
    for en, ex in zip(energy, experience):
        if E <= en:
            k = (en + 1 - E + step_e - 1) // step_e
            hours += k
            E += k * step_e
        if X <= ex:
            k = (ex + 1 - X + step_x - 1) // step_x
            hours += k
            X += k * step_x
        E -= en
        X += ex
    return hours

assert gio_nang_luong(5, [1, 4, 3, 2]) == 6
assert gio_nang_luong(100, [1]) == 0
assert gio_kinh_nghiem(3, [2, 6, 3, 1]) == 2
assert gio_kinh_nghiem(4, [3]) == 0
assert gio_nang_luong(5, [1, 4, 3, 2]) + gio_kinh_nghiem(3, [2, 6, 3, 1]) == Solution().minNumberOfHours(5, 3, [1, 4, 3, 2], [2, 6, 3, 1])
assert min_hours_step(5, 3, [1, 4, 3, 2], [2, 6, 3, 1], 1, 1) == 8
assert min_hours_step(1, 1, [1], [1], 2, 3) == 2          # mỗi bên cần 1 lần tập
assert min_hours_step(1, 1, [3], [4], 2, 3) == 2 + 2        # E cần 4: thiếu 3 -> ceil(3/2)=2; X cần 5: thiếu 4 -> ceil(4/3)=2
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tách hai chỉ số, bù đúng phần thiếu)

```python
class Solution:
    def minNumberOfHours(self, initialEnergy: int, initialExperience: int, energy: List[int], experience: List[int]) -> int:
        e, x, hours = initialEnergy, initialExperience, 0
        for en, ex in zip(energy, experience):
            if e <= en:
                hours += en + 1 - e
                e = en + 1
            if x <= ex:
                hours += ex + 1 - x
                x = ex + 1
            e -= en
            x += ex
        return hours
```

**Checklist**

- [ ] Điều kiện thắng là `>` chặt, nên thiếu khi `cur <= need` và bù `need + 1 - cur`?
- [ ] Mọi phần thiếu đều không âm (`max(0, ...)` hoặc nằm sau `if`)?
- [ ] Sau mỗi trận có `e -= en` và `x += ex` đúng thứ tự?
- [ ] Đã thử `[1]`, `[1]` (hòa ở cả hai) và trường hợp dư thừa (`E = 100`)?
- [ ] Nếu dùng công thức: năng lượng dùng tổng **cả** `n` trận; kinh nghiệm dùng tổng **trước** trận `i`?
- [ ] Signature đúng `minNumberOfHours(self, initialEnergy, initialExperience, energy, experience)` và có `from typing import List`?